# More candidates can expose scorer mistakes

Day 26 · Chapter 15 · CPU worked notebook

Read Chapter15 sections15.1–15.4. Predict: must selected accuracy rise when at least one correct candidate becomes more likely? What if the scorer favors a common wrong answer?

Saved reference preview from the bounded CPU reference execution. It is a fixed result; run the cells to regenerate figures after changing controls. Make the prediction above before inspecting the preview.

![Saved reference 1: selection and sampling](../figures/chapter-15/day-26-02_selection_and_sampling-01.png)

In [ ]:
from pathlib import Path
import sys
root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "dongxi_llms").is_dir())
if str(root / "src") not in sys.path:
    sys.path.insert(0, str(root / "src"))
import torch
import matplotlib.pyplot as plt
torch.set_num_threads(1)
plt.rcParams.update({"figure.figsize": (8, 3.8), "font.size": 11,
                     "axes.spines.top": False, "axes.spines.right": False})
# CPU only. Local original fixtures; no checkpoint/network/server needed.


In [ ]:
from dongxi_llms.distillation_lab import inference_comparison, majority_vote
result = inference_comparison()
rows = result["rows"]
fig, ax = plt.subplots()
for key, label in (("majority_accuracy", "Majority vote"),
                   ("proxy_best_n_accuracy", "Imperfect ranker"),
                   ("oracle_any_correct", "Oracle availability")):
    ax.plot([r["candidate_token_budget"] for r in rows], [r[key] for r in rows],
            marker="o", label=label)
ax.plot([r["candidate_token_budget"] for r in rows],
        [r["oracle_iid_formula"] for r in rows], linestyle=":", color="gray", label="Independent formula")
ax.set(xlabel="Illustrative candidate tokens (8 per candidate)", ylabel="Simulated accuracy / availability")
ax.legend()
plt.show()
print(result["mode"], result["trials"], "trials; categorical draws, not LLM output")

Reference explanation: oracle availability asks whether a correct answer exists; a deployed ranker must identify it. The ranker in this simulator increasingly selects its favored wrong class.

Exercise: test majority ties and estimate rejection cost if acceptance is0.1 rather than0.5.

In [ ]:
assert majority_vote([3, 4, 4, 3]) == 3  # earliest candidate among tied modal answers
candidate_cost = 32
for acceptance in (.5, .1):
    print("Acceptance:", acceptance, "projected tokens per accepted example:",
          candidate_cost/acceptance)
print("Retry limits and prompt difficulty change the accepted data mixture.")

Controlled change: edit the finite simulator's score ordering in a copied experiment, predict each curve and record that new source identity. Evidence boundary: all candidate costs here are illustrative; scores are synthetic; independent draws do not reproduce correlated LLM errors.